# Nhóm 3 — MNIST CNN-LSTM trên Google Colab
Notebook điều hợp dùng chung Dataset, model và runner trong `src/`/`scripts`; không chứa logic huấn luyện riêng.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/BTL03')
PROJECT_ROOT=Path('/content/BTL03')
if not (PROJECT_ROOT/'scripts/run_group3_cnn_lstm.py').is_file(): raise FileNotFoundError('Checkout BTL03 vào /content/BTL03 trước khi chạy.')
import os,sys,torch
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
from src.utils import seed_everything
seed_everything(42)
gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print('Thiết bị:',gpu)
if torch.cuda.is_available() and 'T4' not in gpu: print('Lưu ý: runtime hiện tại không phải GPU T4.')

## T0206 — kiểm tra chuỗi và sinh ảnh minh chứng
Dataset gốc MNIST cần có bốn file `.npy` trong `data/raw/mnist/`; runner có thể dùng IDX downloader hiện có. Smoke-test chỉ dùng ảnh giả và không phải kết quả học thuật.

In [ ]:
!python scripts/run_group3_cnn_lstm.py --dataset-only

In [ ]:
from IPython.display import display,Image
display(Image(filename=str(PROJECT_ROOT/'outputs/figures/t0206_seq_mnist_samples.png')))

## T0207 — huấn luyện 30 epoch
Keras legacy dùng SGD 0.01, MAE, batch 128, 30 epoch. Legacy dùng MNIST test làm `validation_data`; vì vậy kết quả MAE được ghi đúng là split legacy/public-test, không phải holdout độc lập. MAE 0.04 là mục tiêu cần kiểm chứng, không được xem là bảo đảm trước khi chạy.

In [ ]:
!python scripts/run_group3_cnn_lstm.py

In [ ]:
import json
metrics=json.loads((PROJECT_ROOT/'outputs/figures/t0207_cnn_lstm_metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics,ensure_ascii=False,indent=2))
display(Image(filename=str(PROJECT_ROOT/metrics['figure'])))

In [ ]:
import shutil
for relative in ['outputs/checkpoints/03_cnn_lstm','outputs/figures/t0206_seq_mnist_samples.png','outputs/figures/t0207_cnn_lstm_results.png','outputs/figures/t0207_cnn_lstm_metrics.json','experiments/csv_logs','experiments/tensorboard/Group3_CNN_LSTM']:
    source=PROJECT_ROOT/relative
    if source.exists():
        target=DRIVE_ROOT/relative; target.parent.mkdir(parents=True,exist_ok=True)
        if source.is_dir(): shutil.copytree(source,target,dirs_exist_ok=True)
        else: shutil.copy2(source,target)
print('Đã đồng bộ artifacts hiện có lên Drive:',DRIVE_ROOT)